<a href="https://colab.research.google.com/github/jhajagos/SupportingConceptSetGeneration/blob/main/Build_Multimap_ICD10CM_to_OHDSI_Concept.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
OHSDI_VOCABULARY_PATH = "/content/drive/MyDrive/OHDSI/vocabulary/20250922/export/"

In [ ]:
from google.colab import drive

In [ ]:
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import pyspark
spark = pyspark.sql.SparkSession.builder.config("spark.driver.memory", "16g").master("local[*]").getOrCreate()

In [ ]:
concept_sdf = spark.read.parquet(OHSDI_VOCABULARY_PATH + "/concept.parquet")
concept_sdf.createOrReplaceTempView("concept")

print(f"Total rows read: {concept_sdf.count()}")
concept_sdf.limit(10).toPandas()

Total rows read: 7415122


,concept_id,concept_name,domain_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason,vocabulary_id
0,44199121,200 ML Iopamidol 612 MG/ML Injectable Solution...,Drug,Quant Branded Box,None,OMOP3075844,20171006,20230802,U,RxNorm Extension
1,44199122,200 ML Iopamidol 612 MG/ML Injectable Solution...,Drug,Quant Branded Box,None,OMOP3075845,20171006,20230802,U,RxNorm Extension
2,44199123,200 ML Iomeprol 817 MG/ML Injectable Solution ...,Drug,Quant Branded Box,None,OMOP3075846,20171006,20230802,U,RxNorm Extension
3,44199124,200 ML Iomeprol 817 MG/ML Injectable Solution ...,Drug,Quant Branded Box,None,OMOP3075847,20171006,20230802,U,RxNorm Extension
4,44199125,200 ML Iomeprol 714 MG/ML Injectable Solution ...,Drug,Quant Branded Box,None,OMOP3075848,20171006,20230802,U,RxNorm Extension
5,44199126,200 ML Iomeprol 714 MG/ML Injectable Solution ...,Drug,Quant Branded Box,None,OMOP3075849,20171006,20230802,U,RxNorm Extension
6,44199128,200 ML Iohexol 647 MG/ML Injectable Solution [...,Drug,Quant Branded Box,None,OMOP3075851,20171006,20230802,U,RxNorm Extension
7,44199129,200 ML Iohexol 647 MG/ML Injectable Solution [...,Drug,Quant Branded Box,None,OMOP3075852,20171006,20230802,U,RxNorm Extension
8,44199130,200 ML iobitridol 768 MG/ML Injectable Solutio...,Drug,Quant Branded Box,None,OMOP3075853,20171006,20230802,U,RxNorm Extension
9,44199132,200 ML Codeine 21.2 MG/ML / Emetine 0.86 MG/ML...,Drug,Quant Branded Drug,None,OMOP3075855,20171006,20230802,U,RxNorm Extension


In [ ]:
concept_relationship_sdf = spark.read.parquet(OHSDI_VOCABULARY_PATH + "/concept_relationship.parquet")
concept_relationship_sdf.limit(10).toPandas()

,concept_id_1,concept_id_2,valid_start_date,valid_end_date,invalid_reason,relationship_id
0,36017853,36017853,20210402,20991231,None,Maps to
1,36017854,36017854,20210402,20991231,None,Maps to
2,36017861,36017861,20210402,20991231,None,Maps to
3,36017864,36017864,20210402,20991231,None,Maps to
4,36017870,36017870,20210402,20991231,None,Maps to
5,36017874,36017874,20210402,20991231,None,Maps to
6,36017881,36017881,20210402,20991231,None,Maps to
7,36017884,36017884,20210402,20991231,None,Maps to
8,36017885,36017885,20210402,20991231,None,Maps to
9,36017901,36017901,20210402,20991231,None,Maps to


In [ ]:
concept_relationship_sdf.createOrReplaceTempView("concept_relationship")
concept_sdf.createOrReplaceTempView("concept")

In [59]:
icd10cm_mapped_sdf = spark.sql("""
select c1.concept_id, c1.concept_code, c1.concept_name, c1.vocabulary_id,
  c2.concept_id as mapped_concept_id,
  c2.concept_code as mapped_concept_code, c2.concept_name as mapped_concept_name,
  c2.vocabulary_id as mapped_vocabulary_id from concept c1
  join concept_relationship cr on c1.concept_id = cr.concept_id_1 and relationship_id = 'Maps to'
  join concept c2 on c2.concept_id = cr.concept_id_2
  where c1.vocabulary_id = 'ICD10CM' and c2.vocabulary_id = 'SNOMED'
  order by c1.concept_id, c2.concept_id
""")

icd10cm_mapped_sdf.limit(100).toPandas()

,concept_id,concept_code,concept_name,vocabulary_id,mapped_concept_id,mapped_concept_code,mapped_concept_name,mapped_vocabulary_id
0,8689,J11.1,Influenza due to unidentified influenza virus ...,ICD10CM,46273463,10685111000119102,Upper respiratory tract infection caused by In...,SNOMED
1,8690,M76.9,"Unspecified enthesopathy, lower limb, excludin...",ICD10CM,4194889,312836001,Enthesopathy of lower limb,SNOMED
2,8691,O33.7,Maternal care for disproportion due to other f...,ICD10CM,74104,106009009,Fetal condition affecting obstetrical care of ...,SNOMED
3,8691,O33.7,Maternal care for disproportion due to other f...,ICD10CM,80165,31805001,Fetal disproportion,SNOMED
4,8692,Y07.9,Unspecified perpetrator of maltreatment and ne...,ICD10CM,4113020,284607007,Finding relating to aggressive behavior,SNOMED
...,...,...,...,...,...,...,...,...
95,9782,M1A.361,"Chronic gout due to renal impairment, right knee",ICD10CM,46270467,308821000119109,Gout of knee due to renal impairment,SNOMED
96,9783,M1A.362,"Chronic gout due to renal impairment, left knee",ICD10CM,46270467,308821000119109,Gout of knee due to renal impairment,SNOMED
97,9784,M1A.369,"Chronic gout due to renal impairment, unspecif...",ICD10CM,46270467,308821000119109,Gout of knee due to renal impairment,SNOMED
98,9785,M1A.371,"Chronic gout due to renal impairment, right an...",ICD10CM,4035437,239844009,Gout secondary to renal impairment,SNOMED


In [60]:
icd10cm_mapped_df = icd10cm_mapped_sdf.toPandas()

In [62]:
agg_df = icd10cm_mapped_df.groupby(["concept_id","concept_code", "vocabulary_id"])["mapped_concept_code"].agg(lambda x: sorted(list(x)))
agg_df = agg_df.reset_index()
agg_df.columns = ["concept_id","concept_code", "vocabulary_id", "mapped_concept_codes"]

agg_df["n_codes"] = agg_df["mapped_concept_codes"].apply(lambda x: len(x))
agg_df.sort_values("n_codes", ascending=False)

,concept_id,concept_code,vocabulary_id,mapped_concept_codes,n_codes
10200,19904,V13.1,ICD10CM,"[21107000, 214640008, 386662004, 426772002]",4
10199,19903,V13.0,ICD10CM,"[214640008, 386662004, 426772002, 47397001]",4
8526,18229,T45.692,ICD10CM,"[271982007, 276853009, 293331003, 431307001]",4
40576,45541527,T41.292A,ICD10CM,"[271982007, 276853009, 292162001, 431307001]",4
89284,45599542,T47.6X2A,ICD10CM,"[271982007, 276853009, 295400008, 431307001]",4
...,...,...,...,...,...
35618,45535727,S68.719A,ICD10CM,[210644008],1
35617,45535726,S68.615A,ICD10CM,[95855003],1
35616,45535725,S68.614A,ICD10CM,[95855003],1
35615,45535724,S68.610D,ICD10CM,[95855003],1


In [63]:
snomed_terms_mapped_to_df = icd10cm_mapped_df[["mapped_concept_id", "mapped_concept_code", "mapped_concept_name", "mapped_vocabulary_id"]].drop_duplicates()
snomed_terms_mapped_to_df

,mapped_concept_id,mapped_concept_code,mapped_concept_name,mapped_vocabulary_id
0,46273463,10685111000119102,Upper respiratory tract infection caused by In...,SNOMED
1,4194889,312836001,Enthesopathy of lower limb,SNOMED
2,74104,106009009,Fetal condition affecting obstetrical care of ...,SNOMED
3,80165,31805001,Fetal disproportion,SNOMED
4,4113020,284607007,Finding relating to aggressive behavior,SNOMED
...,...,...,...,...
128965,37168537,1284922006,Hemodialysis catheter care,SNOMED
128966,4175555,278150003,Blood group B Rh(D) positive,SNOMED
128974,43020581,471300007,On waiting list for organ transplant,SNOMED
128975,46272734,711149003,Long-term current use of antibiotic,SNOMED


In [64]:
snomed_terms_snomed_list_dict = snomed_terms_mapped_to_df.to_dict("records")
snomed_terms_snomed_dict = {x["mapped_concept_code"]: x for x in snomed_terms_snomed_list_dict}

In [65]:
def map_codes_to_list(codes):
  mapped_code = [snomed_terms_snomed_dict[x] for x in codes]
  return mapped_code

In [67]:
agg_df["mapped_concept_list"] = agg_df["mapped_concept_codes"].apply(lambda x: map_codes_to_list(x))
agg_df.sort_values("concept_code")

,concept_id,concept_code,vocabulary_id,mapped_concept_codes,n_codes,mapped_concept_list
14897,1567237,A00,ICD10CM,[63650001],1,"[{'mapped_concept_id': 198677, 'mapped_concept..."
24141,35205396,A00.0,ICD10CM,[240349003],1,"[{'mapped_concept_id': 4344638, 'mapped_concep..."
24142,35205397,A00.1,ICD10CM,[81020007],1,"[{'mapped_concept_id': 200629, 'mapped_concept..."
24143,35205398,A00.9,ICD10CM,[63650001],1,"[{'mapped_concept_id': 198677, 'mapped_concept..."
14898,1567238,A01,ICD10CM,[302231008],1,"[{'mapped_concept_id': 133685, 'mapped_concept..."
...,...,...,...,...,...,...
29589,35225436,Z99.2,ICD10CM,[105502003],1,"[{'mapped_concept_id': 4019967, 'mapped_concep..."
29590,35225437,Z99.3,ICD10CM,[105503008],1,"[{'mapped_concept_id': 4022073, 'mapped_concep..."
23825,1576313,Z99.8,ICD10CM,[105501005],1,"[{'mapped_concept_id': 437758, 'mapped_concept..."
89990,45600392,Z99.81,ICD10CM,[931000119107],1,"[{'mapped_concept_id': 42873170, 'mapped_conce..."


In [70]:
import json
with open(OHSDI_VOCABULARY_PATH + "/icd10cm_mapped_to_snomed.jsonl", "w") as f:
  for row in agg_df.to_dict("records"):
    f.write(json.dumps(row) + "\n")